In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest

pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 100
np.random.seed(42)

In [ ]:
df = pd.read_csv("telecom_master.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
df.isna().sum()

In [ ]:
features = [
    "avg_monthly_gb",
    "avg_voice_min",
    "tenure_months",
    "arpu",
    "complaints_6m",
    "days_since_last_recharge",
]

X_raw = SimpleImputer(strategy="median").fit_transform(df[features])

In [ ]:
km_raw = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_raw)
raw_centers = pd.DataFrame(km_raw.cluster_centers_, columns=features).round(1)
raw_centers

In [ ]:
X = StandardScaler().fit_transform(X_raw)

In [ ]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    rows.append(
        {
            "k": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(
                X, km.labels_, sample_size=2000, random_state=1
            ),
        }
    )
sweep = pd.DataFrame(rows).round(3)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.bar(sweep.k, sweep.inertia, color="#A9C4D2")
ax1.set_ylabel("Inertia")
ax2 = ax1.twinx()
ax2.plot(sweep.k, sweep.silhouette, color="#F2A03D", marker="o")
ax2.set_ylabel("Silhouette")
ax1.set_xlabel("k")
plt.title("Choosing k: inertia vs. silhouette")
plt.show()
sweep

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
df["segment"] = km.labels_
df["segment"].value_counts().sort_index()

In [ ]:
profile = (
    df.groupby("segment")
    .agg(
        size=("customer_id", "count"),
        mean_arpu=("arpu", "mean"),
        mean_tenure=("tenure_months", "mean"),
        mean_gb=("avg_monthly_gb", "mean"),
        mean_voice=("avg_voice_min", "mean"),
        mean_complaints=("complaints_6m", "mean"),
        churn_rate=("churn", "mean"),
    )
    .round(2)
)
profile["share"] = (profile["size"] / len(df) * 100).round(1)
profile

In [ ]:
print("Plan type mix by segment:")
display(pd.crosstab(df.segment, df.plan_type, normalize="index").round(2))
print("\nDevice type mix by segment:")
display(pd.crosstab(df.segment, df.device_type, normalize="index").round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
profile["mean_arpu"].plot(
    kind="bar", ax=axes[0], color="#4C72B0", title="Mean ARPU by segment"
)
profile["mean_gb"].plot(
    kind="bar", ax=axes[1], color="#55A868", title="Mean data (GB) by segment"
)
profile["churn_rate"].plot(
    kind="bar", ax=axes[2], color="#C44E52", title="Churn rate by segment"
)
for ax in axes:
    ax.set_xlabel("segment")
plt.tight_layout()
plt.show()

In [ ]:
u = pd.read_csv("usage_revenue_monthly.csv")
m = pd.read_csv("telecom_master.csv")
print(u.shape, u.month.nunique(), "months")
u.describe().round(2)

In [ ]:
print("Rows with exactly zero revenue:", (u.revenue_inr == 0).sum())
print("Rows with intl_min > 500:", (u.intl_min > 500).sum())
print("Rows with any failed payments:", (u.failed_payment_count > 0).sum())

In [ ]:
u["intl_share"] = u.intl_min / (u.voice_min + u.intl_min + 1)
u["revenue_per_gb"] = u.revenue_inr / u.data_gb.clip(lower=0.01)
u["zero_revenue"] = (u.revenue_inr == 0).astype(int)
u["data_to_voice"] = u.data_gb / u.voice_min.clip(lower=1)

feat = [
    "data_gb",
    "voice_min",
    "intl_min",
    "sms_count",
    "revenue_inr",
    "failed_payment_count",
    "intl_share",
    "revenue_per_gb",
    "data_to_voice",
]
u[feat].describe().round(2)

In [ ]:
X_anom = StandardScaler().fit_transform(u[feat].replace([np.inf, -np.inf], 0).fillna(0))

for c in [0.005, 0.01, 0.02]:
    iso = IsolationForest(contamination=c, n_estimators=300, random_state=42).fit(
        X_anom
    )
    u[f"flag_{c}"] = (iso.predict(X_anom) == -1).astype(int)
    u[f"score_{c}"] = -iso.score_samples(X_anom)
    print(f'contamination {c:<6} flagged {int(u[f"flag_{c}"].sum()):>4} rows')

In [ ]:
top = (
    u[u["flag_0.01"] == 1]
    .sort_values("score_0.01", ascending=False)
    .head(20)
    .merge(
        m[["customer_id", "plan_type", "tenure_months", "region"]],
        on="customer_id",
        how="left",
    )
)

top[
    [
        "customer_id",
        "month",
        "data_gb",
        "voice_min",
        "intl_min",
        "revenue_inr",
        "failed_payment_count",
        "plan_type",
        "score_0.01",
    ]
]

In [ ]:
def classify(row):
    if row["intl_min"] > 500 and row["data_gb"] < 1 and row["revenue_inr"] < 150:
        return "Bypass / SIM-box fraud"
    if row["failed_payment_count"] >= 2 and row["revenue_inr"] == 0:
        return "Subscription fraud"
    if (
        row["revenue_inr"] == 0
        and row["data_gb"] > 1
        and row["failed_payment_count"] == 0
    ):
        return "Revenue leakage (billing fault)"
    return "Ordinary heavy user (not a problem)"


top["classification"] = top.apply(classify, axis=1)
top["classification"].value_counts()

In [ ]:
top[
    [
        "customer_id",
        "month",
        "data_gb",
        "intl_min",
        "revenue_inr",
        "failed_payment_count",
        "classification",
    ]
]

In [ ]:
WEEKS = 26  # six-month window
BUDGET_PER_WEEK = 45
COST_PER_CASE = 450

for c in [0.005, 0.01, 0.02]:
    n = int(u[f"flag_{c}"].sum())
    per_week = n / WEEKS
    cost = n * COST_PER_CASE
    fits = "fits budget" if per_week <= BUDGET_PER_WEEK else "OVER budget"
    print(
        f"contamination {c}: {n:>4} cases  ~{per_week:5.1f}/week  cost Rs {cost:>8,}  -> {fits}"
    )

In [ ]:
risk_view = profile[["size", "share", "mean_arpu", "churn_rate"]].copy()
risk_view["revenue_at_risk"] = (
    risk_view["size"] * risk_view["churn_rate"] * risk_view["mean_arpu"]
).round(0)
risk_view = risk_view.sort_values("revenue_at_risk", ascending=False)
risk_view

In [ ]:
highest_risk_seg = risk_view.index[0]
highest_churn_seg = profile["churn_rate"].idxmax()
base_churn = profile["churn_rate"].mean()
seg_churn = profile.loc[highest_risk_seg, "churn_rate"]
lift = seg_churn / base_churn if base_churn > 0 else float("nan")

print(
    f"Segment with the single highest raw churn rate: {highest_churn_seg} "
    f'({profile.loc[highest_churn_seg, "churn_rate"]:.0%})'
)
print(
    f"Segment with the highest expected revenue at risk: {highest_risk_seg} "
    f'(churn rate {seg_churn:.0%}, mean ARPU Rs {profile.loc[highest_risk_seg, "mean_arpu"]:.0f})'
)
print(f"Its churn-rate lift vs. the average segment: {lift:.1f}x")

In [ ]:
RETENTION_CALL_COST = (
    50  # Rs per proactive retention contact (illustrative, from MoP 4.1 figures)
)
LOST_SUB_MULTIPLE = 20  # a lost subscriber costs ~20x a retention call, per MoP 4.1

seg_size = int(profile.loc[highest_risk_seg, "size"])
seg_churn_rate = profile.loc[highest_risk_seg, "churn_rate"]

# scale the 6-month churn figure down to one quarter, and assume we can identify
# and target roughly the at-risk subset (not the whole segment)
quarterly_churn_rate = seg_churn_rate / 2
at_risk_subs = seg_size * quarterly_churn_rate

value_per_saved_sub = RETENTION_CALL_COST * LOST_SUB_MULTIPLE  # the 20:1 rule, directly
CHURN_REDUCTION = 0.25  # proactive outreach cuts churn among those contacted by 25%
subs_saved = at_risk_subs * CHURN_REDUCTION

value_protected = subs_saved * value_per_saved_sub
campaign_cost = at_risk_subs * RETENTION_CALL_COST  # call only the at-risk subset
net_value_quarter = value_protected - campaign_cost

print(
    f"Segment: Data-Heavy Young  |  size: {seg_size:,}  |  quarterly churn rate: {quarterly_churn_rate:.1%}"
)
print(f"Subscribers predicted to churn this quarter: {at_risk_subs:,.0f}")
print(
    f"Value of one retained subscriber (20x a Rs{RETENTION_CALL_COST} call): Rs {value_per_saved_sub:,.0f}"
)
print(
    f"If proactive outreach cuts their churn by {CHURN_REDUCTION:.0%}: {subs_saved:,.0f} subscribers saved"
)
print(f"  Value protected: Rs {value_protected:,.0f}")
print(
    f"  Campaign cost ({at_risk_subs:,.0f} targeted calls @ Rs{RETENTION_CALL_COST}): Rs {campaign_cost:,.0f}"
)
print(f"  Net value per quarter: Rs {net_value_quarter:,.0f}")

In [ ]:
baseline = IsolationForest(contamination=0.01, random_state=42).fit(X_anom)
tuned = IsolationForest(
    contamination=0.01, n_estimators=500, max_features=0.8, random_state=42
).fit(X_anom)

base_scores = -baseline.score_samples(X_anom)
tuned_scores = -tuned.score_samples(X_anom)

base_top = set(pd.Series(base_scores).nlargest(20).index)
tuned_top = set(pd.Series(tuned_scores).nlargest(20).index)
overlap = len(base_top & tuned_top)

print(f"Overlap between baseline and tuned top-20 flagged rows: {overlap}/20")
print(
    f"Correlation between baseline and tuned scores: {np.corrcoef(base_scores, tuned_scores)[0, 1]:.3f}"
)